In [1]:
import torch
import numpy as np
import pandas as pd
from haversine import haversine, Unit
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 300

# 1. Load Dataset

In [3]:
trainpath = f'../../../data/top30groups/LongLatCombined/train1/train{partition}.csv'
testpath = f'../../../data/top30groups/LongLatCombined/test1/test{partition}.csv'
traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

In [4]:
combined = pd.concat([traindata, testdata], axis = 0)

### Find unique locations and construct global graph

In [5]:
# Extract unique locations for node creation
combined['location'] = list(zip(combined['longitude'], combined['latitude']))
unique_locations = combined['location'].drop_duplicates().reset_index(drop=True)

# Map locations to an identity
location2id = {loc: idx for idx, loc in enumerate(unique_locations)}
combined['location_id'] = combined['location'].map(location2id)

# Encode labels
le = LabelEncoder()
combined['label'] = le.fit_transform(combined['gname'])

# Get global node features
coords = np.array([list(loc) for loc in unique_locations])  # [1790, 2]
print("Feature Matrix shape: ", coords.shape)

# Standardize features
scaler = StandardScaler()
x_global = scaler.fit_transform(coords)  # standardized features

# Build global edge list using 1km Haversine
edges = []
coords_latlon = [(lat, lon) for lon, lat in unique_locations]
for i in range(len(coords_latlon)):
    for j in range(i + 1, len(coords_latlon)):
        if haversine(coords_latlon[i], coords_latlon[j], Unit.KILOMETERS) <= 1.0:
            edges.append((i, j))
            edges.append((j, i))

global_edge_index = torch.tensor(edges, dtype=torch.long).T  # shape [2, num_edges]


Feature Matrix shape:  (4379, 2)


In [6]:
global_edge_index.shape

torch.Size([2, 1330])

In [7]:
unique_nodes = torch.unique(global_edge_index)
print("Nodes with at least one neighbor: ", len(unique_nodes))

Nodes with at least one neighbor:  705


### Creating subgraphs for each node depending on its neighbors

In [8]:
def get_subgraph(center_id, edge_index, x_global):
    # Get neighbors (indices) of center node
    neighbors = edge_index[1][edge_index[0] == center_id]
    node_ids = torch.cat([torch.tensor([center_id]), neighbors]).unique()

    # Remap node indices locally
    id_map = {old_id.item(): i for i, old_id in enumerate(node_ids)}
    new_edges = []
    for source, destination in zip(*edge_index):
        if source in node_ids and destination in node_ids:
            new_edges.append((id_map[source.item()], id_map[destination.item()]))

    # If no edges exist, add a self-loop
    if len(new_edges) == 0:
        center_local_idx = 0  # only node in subgraph
        new_edges = [(0, 0)]
    else:
        center_local_idx = id_map[center_id.item()]

    sub_x = x_global[node_ids]
    sub_edge_index = torch.tensor(new_edges).T

    return sub_x, sub_edge_index, center_local_idx


### Subgraphs for train

In [9]:
from torch_geometric.data import Data

traindata_list = []
for _, row in traindata.iterrows():
    center_id = location2id[(row['longitude'], row['latitude'])]
    label = le.transform([row['gname']])[0]
    
    x, edge_index, center_idx = get_subgraph(torch.tensor(center_id), global_edge_index, torch.tensor(x_global, dtype=torch.float))
    
    traindata_obj = Data(x=x, edge_index=edge_index, y=torch.tensor(label), center=center_idx)
    traindata_list.append(traindata_obj)


### Subgraphs for test

In [10]:
test_data_list = []
for _, row in testdata.iterrows():
    loc = (row['longitude'], row['latitude'])
    
    # Skip if location not in mapping (just in case)
    if loc not in location2id:
        continue
    
    center_id = location2id[loc]
    label = le.transform([row['gname']])[0]
    
    x, edge_index, center_idx = get_subgraph(
        torch.tensor(center_id),
        global_edge_index,
        torch.tensor(x_global, dtype=torch.float)
    )

    testdata_obj = Data(x=x, edge_index=edge_index, y=torch.tensor(label), center=center_idx)
    test_data_list.append(testdata_obj)

### GCN Model

In [11]:
import torch.nn as nn


class GCN(nn.Module):
    def __init__(self, in_channels, hidden1, hidden2, out_channels, activation_fn=F.relu):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden1)
        self.conv2 = GCNConv(hidden1, hidden2)
        self.classifier = nn.Linear(hidden2, out_channels)
        self.activation_fn = activation_fn

    def forward(self, batch):
        x, edge_index = batch.x, batch.edge_index
        x = self.activation_fn(self.conv1(x, edge_index))
        x = self.activation_fn(self.conv2(x, edge_index))
        center_embeddings = x[batch.ptr[:-1]]
        out = self.classifier(center_embeddings)
        return F.log_softmax(out, dim=1)


### Training and Testing

In [12]:
from sklearn.model_selection import train_test_split

train_set, val_set = train_test_split(traindata_list, test_size=0.2, random_state=42)


### Define batches

In [13]:
def train(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)
    return correct / total

In [14]:
from itertools import product
import random

best_model_state = None
best_val_acc = 0.0
patience = 300

import torch.nn.functional as F

activation_map = {
    'relu': F.relu,
    'tanh': F.tanh
}

param_dist = {
    'hidden1': [10, 50, 100, 150, 200, 300],
    'hidden2': [25, 50, 100],
    'activation': ['relu', 'tanh'],
    'lr': [0.0001, 0.001, 0.01],
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2],
    'batch_size': [128, 256, 512]
}

keys, values = zip(*param_dist.items())
all_combinations = [dict(zip(keys, v)) for v in product(*values)]
random.seed(42)  # for reproducibility
combinations = random.sample(all_combinations, k=100)

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

best_params = None
num_classes = len(le.classes_)

for i, params in enumerate(combinations):
    print(f"Combination {i+1}/{len(combinations)}: {params}")

    act_fn = activation_map[params["activation"]]

    model = GCN(
        in_channels=2,
        hidden1=params["hidden1"],
        hidden2=params["hidden2"],
        out_channels=num_classes,
        activation_fn=act_fn
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["alpha"])
    criterion = torch.nn.CrossEntropyLoss()

    train_loader = DataLoader(train_set, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=params["batch_size"])
    test_loader = DataLoader(test_data_list, batch_size=params["batch_size"])

    patience_counter = 0

    for epoch in range(1, 1500):
        avg_loss = train(train_loader)
        train_acc = evaluate(train_loader)
        val_acc = evaluate(val_loader)

        if epoch % 50 == 0:
            print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_model_state = model.state_dict()
            best_params = params
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    
final_model = GCN(
    in_channels=2,
    hidden1=best_params['hidden1'],
    hidden2=best_params['hidden2'],
    out_channels=num_classes,
    activation_fn=activation_map[best_params['activation']]
).to(device)

final_model.load_state_dict(best_model_state)

test_loader = DataLoader(test_data_list, batch_size=best_params["batch_size"])
test_acc = evaluate(test_loader)

print("\nBest hyperparameters:", best_params)
print(f"Final Test Accuracy after early stopping: {test_acc:.4f}")

Combination 1/100: {'hidden1': 50, 'hidden2': 25, 'activation': 'relu', 'lr': 0.001, 'alpha': 1e-05, 'batch_size': 128}
Epoch 050 | Loss: 0.5677 | Train Acc: 0.7933 | Val Acc: 0.8103
Epoch 100 | Loss: 0.4853 | Train Acc: 0.8246 | Val Acc: 0.8254
Epoch 150 | Loss: 0.4522 | Train Acc: 0.8302 | Val Acc: 0.8397
Epoch 200 | Loss: 0.4282 | Train Acc: 0.8300 | Val Acc: 0.8190
Epoch 250 | Loss: 0.4237 | Train Acc: 0.8266 | Val Acc: 0.8206
Epoch 300 | Loss: 0.3975 | Train Acc: 0.8339 | Val Acc: 0.8286
Epoch 350 | Loss: 0.3913 | Train Acc: 0.8329 | Val Acc: 0.8254
Epoch 400 | Loss: 0.3806 | Train Acc: 0.8409 | Val Acc: 0.8421
Epoch 450 | Loss: 0.3719 | Train Acc: 0.8558 | Val Acc: 0.8460
Epoch 500 | Loss: 0.3664 | Train Acc: 0.8583 | Val Acc: 0.8444
Epoch 550 | Loss: 0.3609 | Train Acc: 0.8621 | Val Acc: 0.8476
Epoch 600 | Loss: 0.3517 | Train Acc: 0.8516 | Val Acc: 0.8452
Epoch 650 | Loss: 0.3511 | Train Acc: 0.8629 | Val Acc: 0.8563
Epoch 700 | Loss: 0.3479 | Train Acc: 0.8546 | Val Acc: 0.853

In [15]:
#Best hyperparameters: {'h1': 10, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 128}
#Final Test Accuracy after early stopping: 0.7556